# Neural Networks — Exercises

Companion to the note [Neural Networks](Neural%20Networks.md).

Practise the feed-forward network $h^{(l)}=\phi(W^{(l)}h^{(l-1)}+b^{(l)})$: why the non-linearity matters,
output layers and losses, counting parameters, universal approximation with ReLUs, and finally building and training
MLPs from scratch in NumPy (no PyTorch needed). **Code convention:** in the math, vectors are columns
($z = Wh + b$); in code, samples are *rows*, so a layer is `Z = H @ W + b` with `W` of shape `(n_in, n_out)`
(this is $W^\top$, and it is exactly sklearn's `coefs_` layout).

**15 exercises** · 🧠 Concept ×3 · ✍️ By hand ×6 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

import warnings
warnings.filterwarnings("ignore")
from scipy.special import expit, log_softmax as _sp_log_softmax, softmax as _sp_softmax
from sklearn.datasets import make_moons, load_digits
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
rng = np.random.default_rng(0)

def relu(z):
    return np.maximum(0.0, z)

## Part A · Concepts

### Exercise 1 · Why do we need the non-linearity?
*🧠 Concept · ★☆☆*

A colleague builds a 10-layer network but forgets the activation function, i.e. $\phi(z)=z$ everywhere.

1. What function class can this network represent? Show it in one line.
2. Can it solve XOR (inputs $\{0,1\}^2$, label $x_1 \oplus x_2$) with a sigmoid output on top? Why/why not?
3. Is the 10-layer linear network *useless* compared with one linear layer, or can depth still change something (e.g. during training)?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Compose two affine maps $W_2(W_1x+b_1)+b_2$ and simplify. Then recall what a single logistic unit's decision boundary looks like.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $W_L(\cdots(W_1x+b_1)\cdots)+b_L = \underbrace{W_L\cdots W_1}_{W}x + \underbrace{(\dots)}_{b}$: a single affine map. Depth adds **no** expressive power.
2. No. Sigmoid on top of an affine map gives a **linear decision boundary** ($w^\top x + b = 0$). XOR's positive points $(0,1),(1,0)$
   and negative points $(0,0),(1,1)$ are not linearly separable ([[Linear Models for Classification]]).
3. It represents the same function class, but the *parametrisation* differs: the loss landscape over $(W_1,\dots,W_L)$ is non-convex and
   gradient descent has different (implicit-regularisation) dynamics, e.g. a bias toward low-rank solutions. Expressively, though, it is just linear.

</details>

### Exercise 2 · Output layer + loss pairing
*🧠 Concept · ★☆☆*

For each task choose the output activation, the number of output units and the loss:

(a) Predict house prices. (b) Spam vs. not spam. (c) Classify an image into one of 100 classes.
(d) Tag a photo with any subset of 20 labels ("dog", "beach", ...). (e) Predict a probability distribution over 5 ratings.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

The note's rule: regression → linear + MSE; binary → sigmoid + BCE; multiclass → softmax + cross-entropy. Which one is (d)?

</details>

<details>
<summary><b>✅ Solution</b></summary>

| Task | Output | Units | Loss |
|---|---|---|---|
| (a) regression | linear (identity) | 1 | MSE (or Huber) |
| (b) binary | sigmoid | 1 | binary cross-entropy |
| (c) multiclass | softmax | 100 | categorical cross-entropy |
| (d) multi-**label** | sigmoid *per label* (independent) | 20 | sum of 20 BCEs |
| (e) distribution | softmax | 5 | cross-entropy with soft targets (= KL up to a constant) |

(d) is the classic trap: softmax forces the labels to compete (probabilities sum to 1), but an image can be "dog" **and** "beach".
These pairings are the canonical link + negative log-likelihood pairs from [[Logistic Regression]], which also make $\delta^{(L)} = \hat y - y$ ([[Backpropagation]]).

</details>

### Exercise 3 · Universal approximation vs. depth
*🧠 Concept · ★★☆*

The universal approximation theorem says one hidden layer suffices to approximate any continuous function on a compact set.

1. Why do we still use deep networks? Give two reasons.
2. What does the theorem **not** say? List at least three things it does not guarantee.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Think about *how many* units are needed, and about learning from finite data with gradient descent.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. **Efficiency**: some functions need exponentially many units with one hidden layer but only polynomially many with depth
   (e.g. a ReLU net of depth $L$ can create $\sim 2^L$ linear pieces by composing "folds", a shallow one needs $\sim 2^L$ units).
   **Hierarchical features**: depth composes simple features into complex ones (edges → parts → objects), which matches the structure of real data and generalises better.
2. It does not say (i) how many units are needed (may be astronomically many), (ii) that gradient descent will *find* the weights,
   (iii) that the network *generalises* from finite samples, (iv) anything outside the compact set (extrapolation).

</details>

## Part B · By hand

### Exercise 4 · Counting parameters
*✍️ By hand · ★☆☆*

An MLP for MNIST has layer sizes $784 \to 256 \to 128 \to 10$ (fully connected, with biases).
How many trainable parameters does it have? How many of them are in the first layer?

In [ ]:
n_params = None
n_first_layer = None

_sizes = [784, 256, 128, 10]
check('total parameters', n_params, sum(a * b + b for a, b in zip(_sizes[:-1], _sizes[1:])))
check('first layer', n_first_layer, np.zeros((784, 256)).size + np.zeros(256).size)

<details>
<summary><b>💡 Hint</b></summary>

A layer $n_{in}\to n_{out}$ has $n_{in}n_{out}$ weights and $n_{out}$ biases.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$784\cdot256+256 = 200{,}960$; $256\cdot128+128 = 32{,}896$; $128\cdot10+10=1{,}290$. Total **235,146**, of which
200,960 (85%) sit in the first layer. This is why images use weight sharing ([[CNN]]): a fully connected first layer scales with the number of pixels.

```python
n_params = 784*256 + 256 + 256*128 + 128 + 128*10 + 10
n_first_layer = 784*256 + 256
```

</details>

### Exercise 5 · A forward pass by hand
*✍️ By hand · ★☆☆*

A network with one ReLU hidden layer and a linear output has
$W^{(1)} = \begin{pmatrix}1&-1\\2&0\end{pmatrix},\ b^{(1)} = \begin{pmatrix}0\\-1\end{pmatrix},\
W^{(2)} = \begin{pmatrix}3&-2\end{pmatrix},\ b^{(2)} = 0.5$.
Compute $z^{(1)}$, $h^{(1)}$ and $\hat y$ for $x = (1, 2)^\top$.

In [ ]:
z1 = None    # list of 2 numbers
h1 = None
yhat = None

_W1 = np.array([[1., -1], [2, 0]]); _b1 = np.array([0., -1]); _W2 = np.array([[3., -2]]); _x = np.array([1., 2])
check('z1', z1, _W1 @ _x + _b1)
check('h1', h1, relu(_W1 @ _x + _b1))
check('yhat', yhat, (_W2 @ relu(_W1 @ _x + _b1) + 0.5)[0])

<details>
<summary><b>💡 Hint</b></summary>

Row $i$ of $W^{(1)}$ times $x$, plus $b_i$. Then clip negatives to 0.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$z^{(1)} = (1\cdot1 - 1\cdot2 + 0,\ 2\cdot1 + 0\cdot2 - 1) = (-1, 1)$, $h^{(1)} = \text{ReLU}(z^{(1)}) = (0, 1)$,
$\hat y = 3\cdot 0 - 2\cdot 1 + 0.5 = -1.5$. Note the first hidden unit is *inactive* for this input, so its weights get no gradient from it ([[Backpropagation]]).

```python
z1 = [-1, 1]
h1 = [0, 1]
yhat = -1.5
```

</details>

### Exercise 6 · Two linear layers collapse
*✍️ By hand · ★☆☆*

Without activation, $\hat y = W^{(2)}(W^{(1)}x + b^{(1)}) + b^{(2)}$ with
$W^{(1)} = \begin{pmatrix}1&2\\0&1\end{pmatrix}, b^{(1)} = \begin{pmatrix}1\\0\end{pmatrix}, W^{(2)} = \begin{pmatrix}1&-1\end{pmatrix}, b^{(2)} = 2$.
Find the equivalent single layer $\hat y = Wx + b$.

In [ ]:
W_eff = None   # e.g. [w1, w2]
b_eff = None

_W1 = np.array([[1., 2], [0, 1]]); _b1 = np.array([1., 0]); _W2 = np.array([[1., -1]])
_X = rng.normal(size=(5, 2))
if W_eff is not None and b_eff is not None:
    check('same function', np.array(W_eff, float).reshape(1, 2) @ _X.T + b_eff, _W2 @ (_W1 @ _X.T + _b1[:, None]) + 2)
else:
    check('same function', None, 0)

<details>
<summary><b>💡 Hint</b></summary>

$W = W^{(2)}W^{(1)}$ and $b = W^{(2)}b^{(1)} + b^{(2)}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$W = (1,-1)\begin{pmatrix}1&2\\0&1\end{pmatrix} = (1, 1)$, $b = (1,-1)\cdot(1,0)^\top + 2 = 3$. So $\hat y = x_1 + x_2 + 3$.

```python
W_eff = [1, 1]
b_eff = 3
```

</details>

### Exercise 7 · Solve XOR with two ReLUs
*✍️ By hand · ★★☆*

Find weights for $\hat y = w^{(2)\top}\,\text{ReLU}(W^{(1)}x + b^{(1)}) + b^{(2)}$ with **2 hidden units** that output exactly
$0, 1, 1, 0$ on $x = (0,0), (1,0), (0,1), (1,1)$.

In [ ]:
W1 = None   # 2x2, math convention (z = W1 @ x + b1)
b1 = None   # length 2
w2 = None   # length 2
b2 = None

_Xx = np.array([[0, 0], [1, 0], [0, 1], [1, 1]], float)
if None in (W1, b1, w2, b2):
    check('XOR outputs', None, 0)
else:
    _out = relu(_Xx @ np.array(W1, float).T + np.array(b1, float)) @ np.array(w2, float) + b2
    check('XOR outputs', _out, [0, 1, 1, 0])

<details>
<summary><b>💡 Hint 1</b></summary>

Both hidden units can look at $s = x_1 + x_2 \in \{0,1,2\}$. You need a function of $s$ that is $0,1,0$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$\text{ReLU}(s) - 2\,\text{ReLU}(s-1)$ evaluated at $s=0,1,2$?

</details>

<details>
<summary><b>✅ Solution</b></summary>

Take $h_1 = \text{ReLU}(x_1+x_2)$, $h_2 = \text{ReLU}(x_1+x_2-1)$, $\hat y = h_1 - 2h_2$:

| $x$ | $s$ | $h_1$ | $h_2$ | $\hat y$ |
|---|---|---|---|---|
| (0,0) | 0 | 0 | 0 | 0 |
| (1,0),(0,1) | 1 | 1 | 0 | 1 |
| (1,1) | 2 | 2 | 1 | 0 |

The hidden layer maps the four points to $(0,0), (1,0), (1,0), (2,1)$, where they *are* linearly separable. That is what a hidden layer does: learn a representation.

```python
W1 = [[1, 1], [1, 1]]
b1 = [0, -1]
w2 = [1, -2]
b2 = 0
```

</details>

### Exercise 8 · Softmax cross-entropy by hand
*✍️ By hand · ★★☆*

A 3-class network outputs logits $z = (2, 1, 0)$ and the true class is $k=0$ (the first).
Compute the softmax probabilities $\hat p$ and the loss $L = -\log \hat p_0$. Then show that $L = \log(1 + e^{-1} + e^{-2})$.

In [ ]:
p_hand = None   # 3 probabilities
loss_hand = None

check('softmax', p_hand, _sp_softmax([2., 1, 0]), tol=1e-3)
check('loss', loss_hand, -_sp_log_softmax([2., 1, 0])[0], tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

$\hat p_k = e^{z_k}/\sum_j e^{z_j}$ with $e^2\approx7.389$, $e\approx2.718$, $e^0=1$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\sum_j e^{z_j} = 7.389+2.718+1 = 11.107$, so $\hat p = (0.665, 0.245, 0.090)$ and $L = -\log 0.665 = 0.408$.
Dividing numerator and denominator by $e^2$: $\hat p_0 = 1/(1+e^{-1}+e^{-2})$, hence $L = \log(1+e^{-1}+e^{-2}) = 0.4076$.
This "log-sum-exp minus the correct logit" form is how stable implementations compute it (Exercise 11).

```python
p_hand = [0.665, 0.245, 0.090]
loss_hand = 0.4076
```

</details>

### Exercise 9 · A tent function from three ReLUs
*✍️ By hand · ★★★*

Write the tent function $f(x) = \max(0, 1-|x|)$ **exactly** as a one-hidden-layer ReLU network
$f(x) = \sum_{k=1}^3 a_k\,\text{ReLU}(x - c_k)$ (all input weights 1). Give $c$ and $a$.
Why does this, together with sums of shifted/scaled tents, prove a 1-D version of universal approximation?

In [ ]:
c = None   # 3 kinks
a = None   # 3 output weights

_xs = np.linspace(-3, 3, 601)
if c is None or a is None:
    check('tent function', None, 0)
else:
    check('tent function', relu(_xs[:, None] - np.array(c, float)) @ np.array(a, float), np.maximum(0, 1 - np.abs(_xs)))

<details>
<summary><b>💡 Hint 1</b></summary>

A sum of ReLUs is piecewise linear; each $a_k$ *changes the slope* by $a_k$ at the kink $c_k$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

The slopes of the tent are $0, +1, -1, 0$ on $(-\infty,-1), (-1,0), (0,1), (1,\infty)$. What slope changes are needed at $-1, 0, 1$?

</details>

<details>
<summary><b>✅ Solution</b></summary>

Kinks at $c = (-1, 0, 1)$, slope changes $+1, -2, +1$: $f(x) = \text{ReLU}(x+1) - 2\,\text{ReLU}(x) + \text{ReLU}(x-1)$.
For $x>1$ the slopes cancel ($1-2+1=0$) and the value is $(x+1)-2x+(x-1) = 0$. ✔

Any continuous $g$ on $[a,b]$ is uniformly continuous, so the piecewise-linear interpolant through $g(t_k)$ on a fine grid
(a sum of scaled tents, i.e. a ReLU network with one hidden layer) is within any $\varepsilon$ of $g$ once the grid is fine enough.
That is universal approximation in 1-D (see Exercise 12).

```python
c = [-1, 0, 1]
a = [1, -2, 1]
```

</details>

## Part C · Code from scratch

### Exercise 10 · Forward pass of an MLP (checked against sklearn)
*💻 Code · ★☆☆*

Implement `mlp_forward(X, Ws, bs)` for a ReLU MLP with a **sigmoid** output unit, returning $P(y=1\mid x)$ for every row of `X`.
`Ws[l]` has shape `(n_in, n_out)` and `bs[l]` shape `(n_out,)` (row convention). The cell trains an sklearn `MLPClassifier`; plug its
`coefs_` / `intercepts_` into your function and you should reproduce `predict_proba` exactly.

In [ ]:
def mlp_forward(X, Ws, bs):
    # TODO: ReLU on all hidden layers, sigmoid on the last layer; return a 1-D array
    return None

X_m, y_m = make_moons(200, noise=0.2, random_state=0)
sk_mlp = MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=800, random_state=0).fit(X_m, y_m)
p_mine = mlp_forward(X_m, sk_mlp.coefs_, sk_mlp.intercepts_)

check('matches sklearn predict_proba', p_mine, sk_mlp.predict_proba(X_m)[:, 1])

<details>
<summary><b>💡 Hint</b></summary>

Loop over the layers: `H = relu(H @ W + b)` for all but the last, then `expit(H @ W + b).ravel()`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The forward pass is nothing but alternating matrix products and element-wise non-linearities; sklearn uses exactly this with `activation='relu'` and a logistic output for binary problems.

```python
def mlp_forward(X, Ws, bs):
    H = X
    for W, b in zip(Ws[:-1], bs[:-1]):
        H = relu(H @ W + b)
    return expit(H @ Ws[-1] + bs[-1]).ravel()

X_m, y_m = make_moons(200, noise=0.2, random_state=0)
sk_mlp = MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=800, random_state=0).fit(X_m, y_m)
p_mine = mlp_forward(X_m, sk_mlp.coefs_, sk_mlp.intercepts_)
```

</details>

### Exercise 11 · Numerically stable softmax cross-entropy
*💻 Code · ★☆☆*

Implement `log_softmax(Z)` (row-wise) and `cross_entropy(Z, y)` = mean of $-\log \hat p_{n,y_n}$ for logits `Z` of shape `(N, K)` and integer labels `y`.
It must stay finite for huge logits like `Z * 1000` (the naive `np.log(np.exp(Z)/...)` overflows).

In [ ]:
def log_softmax(Z):
    return None

def cross_entropy(Z, y):
    return None

Z_test = rng.normal(size=(6, 4)); y_test = rng.integers(0, 4, size=6)

check('log_softmax', log_softmax(Z_test), _sp_log_softmax(Z_test, axis=1))
check('log_softmax with huge logits', log_softmax(1000 * Z_test), _sp_log_softmax(1000 * Z_test, axis=1))
check('cross-entropy', cross_entropy(Z_test, y_test), -_sp_log_softmax(Z_test, axis=1)[np.arange(6), y_test].mean())

<details>
<summary><b>💡 Hint</b></summary>

Subtract the row maximum first: $\log\hat p_k = (z_k - m) - \log\sum_j e^{z_j - m}$ (log-sum-exp trick). Softmax is shift-invariant.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Subtracting $m=\max_j z_j$ makes every exponent $\le 0$, so nothing overflows, and the largest term is $e^0=1$ so the sum never underflows to 0.
Frameworks fuse softmax and cross-entropy for exactly this reason (`CrossEntropyLoss` takes logits, not probabilities).

```python
def log_softmax(Z):
    Zs = Z - Z.max(axis=1, keepdims=True)
    return Zs - np.log(np.exp(Zs).sum(axis=1, keepdims=True))

def cross_entropy(Z, y):
    return -log_softmax(Z)[np.arange(len(y)), y].mean()

Z_test = rng.normal(size=(6, 4)); y_test = rng.integers(0, 4, size=6)
```

</details>

### Exercise 12 · Universal approximation, constructively
*💻 Code · ★★☆*

Generalise Exercise 9: write `relu_interpolant(knots, values)` that returns a function `f(x)` implemented as
$f(x) = s_0 + \sum_{k=0}^{K-1} a_k\,\text{ReLU}(x - t_k)$ that linearly interpolates the points $(t_k, s_k)$ on $[t_0, t_K]$.
Use it to approximate $\sin(x)$ on $[0, 2\pi]$ with $K = 4, 8, 16, 32$ pieces, and store the max error on a fine grid in `max_err` (dict $K\mapsto$ error).
How does the error scale with $K$?

In [ ]:
def relu_interpolant(knots, values):
    # TODO: slopes of each piece -> a_0 = slope_0, a_k = slope_k - slope_{k-1}
    return None

x_grid = np.linspace(0, 2 * np.pi, 2001)
max_err = None   # {4: ..., 8: ..., 16: ..., 32: ...}

_t = np.linspace(0, 3, 7); _s = rng.normal(size=7)
_f = relu_interpolant(_t, _s)
check('interpolates like np.interp', None if _f is None else _f(np.linspace(0, 3, 50)), np.interp(np.linspace(0, 3, 50), _t, _s))
if max_err is not None:
    check('error shrinks ~4x per doubling of K', max_err[16] / max_err[32] > 3.5 and max_err[4] > max_err[8] > max_err[16], True)
else:
    check('error shrinks ~4x per doubling of K', None, True)

<details>
<summary><b>💡 Hint</b></summary>

`slopes = np.diff(values) / np.diff(knots)`; `a = np.r_[slopes[0], np.diff(slopes)]`; then `f = lambda x: values[0] + relu(x[:, None] - knots[:-1]) @ a`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Each hidden unit adds one kink; $K$ pieces need $K$ hidden units. For a smooth function the linear-interpolation error is
$\le \frac{h^2}{8}\max|f''|$ with $h = 2\pi/K$, so it falls by $\approx 4\times$ per doubling of $K$ (e.g. $K=32$: $\approx 0.005$).
In $d$ dimensions a grid needs $\sim K^d$ units: the curse of dimensionality, and a reason why depth and learned features matter.

```python
def relu_interpolant(knots, values):
    knots = np.asarray(knots, float); values = np.asarray(values, float)
    slopes = np.diff(values) / np.diff(knots)
    a = np.r_[slopes[0], np.diff(slopes)]
    return lambda x: values[0] + relu(np.asarray(x, float)[:, None] - knots[:-1]) @ a

x_grid = np.linspace(0, 2 * np.pi, 2001)
max_err = {}
for K in (4, 8, 16, 32):
    t = np.linspace(0, 2 * np.pi, K + 1)
    max_err[K] = np.max(np.abs(relu_interpolant(t, np.sin(t))(x_grid) - np.sin(x_grid)))
print(max_err)
```

</details>

### Exercise 13 · Train a 2-layer network on two moons
*💻 Code · ★★☆*

Implement `train_mlp(X, y, hidden=16, act='tanh', lr=0.5, epochs=2000, seed=0)` for binary classification:
one hidden layer (activation `'tanh'` or `'linear'`), a sigmoid output and the mean BCE loss, trained with full-batch gradient descent.
Return `(params, losses)` where `params = (W1, b1, W2, b2)`, plus a `predict(params, X, act)` function returning 0/1 labels.
Reach **> 95 % training accuracy** on the moons data.

In [ ]:
def train_mlp(X, y, hidden=16, act='tanh', lr=0.5, epochs=2000, seed=0):
    # TODO: init W1 ~ N(0, 1/d), W2 ~ N(0, 1/hidden), zeros for biases; loop forward/backward/update
    return None

def predict(params, X, act='tanh'):
    return None

X_tr, y_tr = make_moons(200, noise=0.2, random_state=1)
result = train_mlp(X_tr, y_tr)
acc_moons = None if result is None else np.mean(predict(result[0], X_tr) == y_tr)

check('training accuracy > 0.95', None if acc_moons is None else acc_moons > 0.95, True)

<details>
<summary><b>💡 Hint 1</b></summary>

Forward: $Z_1 = XW_1+b_1$, $H=\tanh Z_1$, $p = \sigma(HW_2+b_2)$. Backward (see [[Backpropagation]]): $\delta_2 = (p-y)/N$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$\nabla W_2 = H^\top\delta_2$, $\nabla b_2 = \sum\delta_2$, $\delta_1 = (\delta_2 W_2^\top)\odot(1-H^2)$, $\nabla W_1 = X^\top\delta_1$, $\nabla b_1 = \sum \delta_1$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Sigmoid + BCE gives the clean output error $\delta_2 = \hat p - y$ (divided by $N$ for the mean). The rest is the chain rule with
$\tanh'(z) = 1-\tanh^2 z$. Accuracy on the moons is $\approx 97$–$98\%$ (noise makes 100% unrealistic without overfitting).

```python
def _act(Z, act):
    return np.tanh(Z) if act == 'tanh' else Z

def train_mlp(X, y, hidden=16, act='tanh', lr=0.5, epochs=2000, seed=0):
    r = np.random.default_rng(seed)
    N, d = X.shape
    W1 = r.normal(0, 1 / np.sqrt(d), (d, hidden)); b1 = np.zeros(hidden)
    W2 = r.normal(0, 1 / np.sqrt(hidden), (hidden, 1)); b2 = np.zeros(1)
    y = y.reshape(-1, 1).astype(float)
    losses = []
    for _ in range(epochs):
        H = _act(X @ W1 + b1, act)
        p = expit(H @ W2 + b2)
        losses.append(-np.mean(y * np.log(p + 1e-12) + (1 - y) * np.log(1 - p + 1e-12)))
        d2 = (p - y) / N
        gW2, gb2 = H.T @ d2, d2.sum(0)
        d1 = (d2 @ W2.T) * ((1 - H ** 2) if act == 'tanh' else 1.0)
        gW1, gb1 = X.T @ d1, d1.sum(0)
        W1 -= lr * gW1; b1 -= lr * gb1; W2 -= lr * gW2; b2 -= lr * gb2
    return (W1, b1, W2, b2), losses

def predict(params, X, act='tanh'):
    W1, b1, W2, b2 = params
    return (expit(_act(X @ W1 + b1, act) @ W2 + b2).ravel() > 0.5).astype(int)

X_tr, y_tr = make_moons(200, noise=0.2, random_state=1)
result = train_mlp(X_tr, y_tr)
acc_moons = None if result is None else np.mean(predict(result[0], X_tr) == y_tr)
print('accuracy', acc_moons, 'final loss', result[1][-1])
```

</details>

### Exercise 14 · Experiment: linear vs. non-linear hidden layer on XOR
*💻 Code · ★★☆*

Generate a noisy XOR dataset (4 Gaussian blobs at the corners of $[-1,1]^2$, label = 1 when $x_1x_2<0$, 400 points, std 0.3).
Train your `train_mlp` with `act='linear'` and with `act='tanh'` and store the training accuracies in `acc_lin` and `acc_tanh`.
Explain the result using Exercise 1.

In [ ]:
acc_lin = None
acc_tanh = None

if acc_lin is None:
    check('linear fails, tanh succeeds', None, True)
else:
    check('linear fails, tanh succeeds', acc_lin < 0.75 and acc_tanh > 0.95, True)

<details>
<summary><b>💡 Hint</b></summary>

`centers = np.array([[1,1],[-1,-1],[1,-1],[-1,1]])`; draw 100 points around each, then `y = (X[:,0]*X[:,1] < 0)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With a linear hidden layer the whole network is logistic regression on $x$: a straight decision boundary, accuracy near chance ($\approx 50$–$60\%$).
With tanh the hidden layer bends the space and accuracy is $\approx 97\%$+. The *only* difference is $\phi$.

```python
_r = np.random.default_rng(3)
centers = np.array([[1, 1], [-1, -1], [1, -1], [-1, 1]], float)
X_xor = np.concatenate([c + 0.3 * _r.normal(size=(100, 2)) for c in centers])
y_xor = (X_xor[:, 0] * X_xor[:, 1] < 0).astype(int)
p_lin, _ = train_mlp(X_xor, y_xor, act='linear')
p_tanh, _ = train_mlp(X_xor, y_xor, act='tanh')
acc_lin = np.mean(predict(p_lin, X_xor, 'linear') == y_xor)
acc_tanh = np.mean(predict(p_tanh, X_xor, 'tanh') == y_xor)
print(acc_lin, acc_tanh)
```

</details>

### Exercise 15 · A softmax MLP for handwritten digits
*💻 Code · ★★★*

Train a one-hidden-layer MLP (64 tanh units, softmax output, cross-entropy) on sklearn's `load_digits` (8×8 images, 10 classes)
with full-batch gradient descent. Use the split below, scale pixels to $[0,1]$, and store the **test** accuracy in `acc_digits`.
Compare with `MLPClassifier(hidden_layer_sizes=(64,))`. Also check the debugging rule from [[Training Tricks]]: the initial loss should be $\approx\log 10$.

In [ ]:
Xd, yd = load_digits(return_X_y=True)
Xd = Xd / 16.0
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.3, random_state=0)
acc_digits = None
initial_loss = None

check('test accuracy > 0.93', None if acc_digits is None else acc_digits > 0.93, True)
check('initial loss ~ log(10)', initial_loss, np.log(10), tol=0.15)

<details>
<summary><b>💡 Hint 1</b></summary>

With one-hot $Y$: $\delta_2 = (\text{softmax}(Z_2) - Y)/N$. The rest is identical to Exercise 13.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Use small initial weights (e.g. std $1/\sqrt{n_{in}}$) so the initial logits are near zero; lr ≈ 0.5, ~1500 epochs.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Test accuracy ends around 97%, on par with sklearn's `MLPClassifier` (which uses Adam and mini-batches, see [[Optimizers]]).
With near-zero logits all classes get probability $\approx 1/10$, so the initial loss is $\approx \log 10 = 2.303$; a very different value signals a bug or a bad init.

```python
Xd, yd = load_digits(return_X_y=True)
Xd = Xd / 16.0
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.3, random_state=0)

_r = np.random.default_rng(0)
N, d, Hn, K = len(Xd_tr), 64, 64, 10
W1 = _r.normal(0, 1 / np.sqrt(d), (d, Hn)); b1 = np.zeros(Hn)
W2 = _r.normal(0, 0.01, (Hn, K)); b2 = np.zeros(K)
Y = np.eye(K)[yd_tr]
for epoch in range(1500):
    H = np.tanh(Xd_tr @ W1 + b1)
    Z = H @ W2 + b2
    P = _sp_softmax(Z, axis=1)
    if epoch == 0:
        initial_loss = -np.mean(np.log(P[np.arange(N), yd_tr]))
    d2 = (P - Y) / N
    d1 = (d2 @ W2.T) * (1 - H ** 2)
    W2 -= 0.5 * H.T @ d2; b2 -= 0.5 * d2.sum(0)
    W1 -= 0.5 * Xd_tr.T @ d1; b1 -= 0.5 * d1.sum(0)
acc_digits = np.mean((np.tanh(Xd_te @ W1 + b1) @ W2 + b2).argmax(1) == yd_te)
sk = MLPClassifier(hidden_layer_sizes=(64,), max_iter=500, random_state=0).fit(Xd_tr, yd_tr)
print('mine', acc_digits, 'sklearn', sk.score(Xd_te, yd_te), 'initial loss', initial_loss)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Neural Networks](Neural%20Networks.md).*